# 05 · Ingestão Bronze — physical_itens_venda_caixa
**Squad 3 – Batch | Alexandro Ferreira da Silva**

Primeira camada da arquitetura Medallion. Lê o CSV histórico direto do ADLS2,
adiciona colunas de auditoria e salva como Delta particionado por ano/mês.

### Fluxo


### Colunas de auditoria adicionadas
| Coluna | Conteúdo |
|--------|----------|
| `bronze_ingested_at` | timestamp da ingestão |
| `bronze_source_file` | caminho do arquivo CSV de origem |
| `ano` | extraído de dt_venda (via join com physical_vendas_caixa) |
| `mes` | extraído de dt_venda (via join com physical_vendas_caixa) |

In [0]:
from dotenv import load_dotenv
import os

load_dotenv("/Workspace/Users/alexandrofs31@gmail.com/.env")

CLIENT_ID       = os.getenv("ADLS_CLIENT_ID")
TENANT_ID       = os.getenv("ADLS_TENANT_ID")
CLIENT_SECRET   = os.getenv("ADLS_CLIENT_SECRET")
STORAGE_ACCOUNT = os.getenv("ADLS_STORAGE_ACCOUNT", "internshipdatalake")
CONTAINER       = os.getenv("ADLS_CONTAINER", "raw")

print("✅ Credenciais carregadas.")

In [0]:
from pyspark.sql import functions as F
from datetime import datetime

# Caminhos
TABELA          = "physical_itens_venda_caixa"
TABELA_VENDAS   = "physical_vendas_caixa"
CAMINHO_CSV     = f"batch-data/{TABELA}.csv"
CAMINHO_VENDAS  = f"batch-data/{TABELA_VENDAS}.csv"
CAMINHO_BRONZE  = f"abfss://{CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/bronze/squad3/{TABELA}/"

# OAuth — mesmo dicionário do notebook de setup
opcoes_adls = {
    f"fs.azure.account.auth.type.{STORAGE_ACCOUNT}.dfs.core.windows.net": "OAuth",
    f"fs.azure.account.oauth.provider.type.{STORAGE_ACCOUNT}.dfs.core.windows.net": "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
    f"fs.azure.account.oauth2.client.id.{STORAGE_ACCOUNT}.dfs.core.windows.net": CLIENT_ID,
    f"fs.azure.account.oauth2.client.secret.{STORAGE_ACCOUNT}.dfs.core.windows.net": CLIENT_SECRET,
    f"fs.azure.account.oauth2.client.endpoint.{STORAGE_ACCOUNT}.dfs.core.windows.net": f"https://login.microsoftonline.com/{TENANT_ID}/oauth2/token",
}

print(f"📂 Origem  : abfss://{CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/{CAMINHO_CSV}")
print(f"📦 Destino : {CAMINHO_BRONZE}")

In [0]:
url_itens  = f"abfss://{CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/{CAMINHO_CSV}"
url_vendas = f"abfss://{CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/{CAMINHO_VENDAS}"

df_itens = (spark.read
    .options(**opcoes_adls)
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(url_itens))

# Lê apenas id_transacao + dt_venda de vendas para join de partição
df_vendas = (spark.read
    .options(**opcoes_adls)
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(url_vendas)
    .select("id_transacao", "dt_venda"))

print(f"✅ {TABELA}        : {df_itens.count():>10,} linhas | {len(df_itens.columns)} colunas")
print(f"✅ {TABELA_VENDAS} : {df_vendas.count():>10,} linhas | 2 colunas (id_transacao, dt_venda)")

In [0]:
timestamp_ingestao = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

df_bronze = (df_itens
    # join para trazer dt_venda (necessária para particionamento)
    .join(df_vendas, on="id_transacao", how="left")
    # colunas de partição
    .withColumn("ano", F.year(F.col("dt_venda").cast("date")))
    .withColumn("mes", F.month(F.col("dt_venda").cast("date")))
    # colunas de auditoria Bronze
    .withColumn("bronze_ingested_at", F.lit(timestamp_ingestao).cast("timestamp"))
    .withColumn("bronze_source_file", F.lit(url_itens))
    # remove dt_venda (fica na camada Silver, quando for necessária)
    .drop("dt_venda")
)

print("✅ Colunas finais Bronze:")
for c in df_bronze.columns:
    print(f"   • {c}")

In [0]:
# Célula 5 — Salvar Bronze no Unity Catalog (Delta gerenciado)
(df_bronze
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("ano", "mes")
    .saveAsTable("workspace.squad3.bronze_physical_itens_venda_caixa"))

total = df_bronze.count()
print(f"✅ Bronze salvo no Unity Catalog!")
print(f"   Tabela  : workspace.squad3.bronze_physical_itens_venda_caixa")
print(f"   Linhas  : {total:,}")
print(f"   Partições: ano / mes")

In [0]:
# Cell 6 — Validação do Bronze
df_val = spark.table("workspace.squad3.bronze_physical_itens_venda_caixa")

total_linhas = df_val.count()
colunas = df_val.columns
particoes = df_val.select("ano", "mes").distinct().orderBy("ano", "mes")

print(f"✅ Total de linhas: {total_linhas:,}")
print(f"✅ Total de colunas: {len(colunas)}")
print(f"✅ Colunas: {colunas}")
print(f"\n📅 Partições disponíveis (ano/mes):")
particoes.show(truncate=False)

print("\n📋 Amostra dos dados:")
df_val.show(3, truncate=False)

print("\n🔍 Schema:")
df_val.printSchema()